# QUERY 3
Cuenta de origen y monto de transacciones USD en el período [2022-09-06, 2022-09-15]
con monto menor a 1 centésimo del promedio encontrado para el mismo formato de
pago en el período [2022-09-01, 2022-09-05]

In [ ]:
import pandas as pd

RUTA_DE_DATASETS             = "../../datasets"
NOMBRE_DATASET      = "LI-Small_Trans.csv"
# NOMBRE_DATASET      = "transacciones_sample.csv"
RUTA_RESULTADO_QUERY3        = "q3_solucion.csv"

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_DATASET}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)
# Filtrar USD antes de samplear
transacciones = transacciones_completas[
    transacciones_completas["Payment Currency"] == "US Dollar"
]

In [ ]:
transacciones["Timestamp"]   = pd.to_datetime(transacciones["Timestamp"])
transacciones["Amount Paid"] = pd.to_numeric(transacciones["Amount Paid"], errors="coerce")
transacciones = transacciones.dropna(subset=["Amount Paid", "Payment Format"])

periodo_temprano = transacciones[
    (transacciones["Timestamp"] >= "2022-09-01") &
    (transacciones["Timestamp"] < "2022-09-06")
]
periodo_tardio = transacciones[
    (transacciones["Timestamp"] >= "2022-09-06") &
    (transacciones["Timestamp"] < "2022-09-16")
]

print(f"Período temprano (1/9-5/9): {len(periodo_temprano)} transacciones")
print(f"Período tardío  (6/9-15/9): {len(periodo_tardio)} transacciones")

In [ ]:
import hashlib

def obtener_id_shard(valor, total_shards):
    valor_norm = str(valor).strip()
    hash_hex = hashlib.md5(valor_norm.encode('utf-8')).hexdigest()
    return (int(hash_hex, 16) % total_shards) + 1

formatos = ["ACH", "Wire", "Cheque", "Cash", "Credit Card", "Reinvestment"]
for f in formatos:
    print(f"{f} -> shard {obtener_id_shard(f, 3)}")

In [ ]:
stats_por_formato = (
    periodo_temprano
    .groupby("Payment Format")["Amount Paid"]
    .agg(suma="sum", count="count")
)
stats_por_formato["promedio"] = stats_por_formato["suma"] / stats_por_formato["count"]

print(stats_por_formato)

In [ ]:
df = periodo_tardio.copy()
df["Promedio Formato"] = df["Payment Format"].map(stats_por_formato["promedio"])
df = df.dropna(subset=["Promedio Formato"])

resultado_query3 = df[
    df["Amount Paid"] < df["Promedio Formato"] * 0.01
][["Account", "Amount Paid"]].rename(columns={"Account": "From Account"}).reset_index(drop=True)

guardar_csv(resultado_query3, RUTA_RESULTADO_QUERY3)
print(f"Resultados Q3: {len(resultado_query3)} transacciones")
resultado_query3.head()

In [ ]:
# Validación del resultado
df_validacion = resultado_query3.copy()
df_validacion = df_validacion.merge(
    periodo_tardio[["Account", "Payment Format", "Timestamp", "Amount Paid"]].rename(columns={"Account": "From Account"}),
    on=["From Account", "Amount Paid"],
    how="left"
)
df_validacion["Promedio Formato"] = df_validacion["Payment Format"].map(stats_por_formato["promedio"])
df_validacion["1% Promedio"] = df_validacion["Promedio Formato"] * 0.01
df_validacion["Es valido"] = df_validacion["Amount Paid"] < df_validacion["1% Promedio"]

print(f"Filas totales: {len(df_validacion)}")
print(f"Filas válidas: {df_validacion['Es valido'].sum()}")
print(f"Filas inválidas: {(~df_validacion['Es valido']).sum()}")
df_validacion.head(10)

In [ ]:
print(f"Período temprano total: {len(periodo_temprano)}")
print(f"Período tardío total: {len(periodo_tardio)}")
print("\nTemprano por formato:")
print(periodo_temprano.groupby("Payment Format")["Amount Paid"].count())
print("\nTardío por formato:")
print(periodo_tardio.groupby("Payment Format")["Amount Paid"].count())